# BTCPredictor2 - GPU Training

## Before running
1. Enable GPU: **Runtime -> Change runtime type -> T4 GPU**
2. Run Cell 1 (clones repo + installs packages)
3. Run Cell 2 (uploads btc_merged_features.csv — a file picker will appear)
4. Run Cells 3, 4, 5 to train each model

**Estimated time on T4 GPU:**
- TFT: 3-5 hours
- BiLSTM: 1-2 hours  
- Meta: 5 minutes

In [ ]:
# ============================================================
# CELL 1 - SETUP
# Clones repo from GitHub, installs packages, checks GPU.
# ============================================================

# Clone the repo
GITHUB_URL = 'https://github.com/chefo919/BTCPredictor2.git'
!git clone {GITHUB_URL} /content/BTCPredictor2

import os, sys
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

# Install packages not included in Colab by default
!pip install -q xgboost joblib ta scikit-learn-intelex

# Create folders
os.makedirs('/content/BTCPredictor2/data', exist_ok=True)
os.makedirs('/content/BTCPredictor2/models/saved', exist_ok=True)

# Check GPU and enable mixed precision
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f'GPU: {gpus[0].name}  |  Mixed precision: ON')
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > GPU and restart.')

# A100 has 40 GB VRAM — can use large batches
# VSN accumulates instead of stacking so VRAM usage is minimal regardless
# Attention at batch=512: [512, 4, 1440, 1440] float16 = 6.8 GB — fits in A100 40 GB
import config
config.BATCH_TFT    = 512
config.BATCH_BILSTM = 1024

print()
print(f'TFT:    {config.SEQ_LEN_TFT//60}h lookback -> {config.HORIZON_TFT//60}h prediction | batch={config.BATCH_TFT}')
print(f'BiLSTM: {config.SEQ_LEN_BILSTM//60}h lookback -> {config.HORIZON_BILSTM}min prediction | batch={config.BATCH_BILSTM}')
print()
print('Done. Run Cell 2 to link the training data.')

In [ ]:
# CELL 2 - MOUNT DRIVE AND LINK DATA FILE
from google.colab import drive
drive.mount('/content/drive')

import os

# Show what's in your Drive root to confirm the path
print('Contents of MyDrive:')
for item in sorted(os.listdir('/content/drive/MyDrive')):
    print(f'  {item}')
print()

# Try both possible locations automatically
candidates = [
    '/content/drive/MyDrive/BTCPredictor2/btc_merged_features.csv',
    '/content/drive/MyDrive/btc_merged_features.csv',
]
csv_drive = None
for path in candidates:
    if os.path.exists(path):
        csv_drive = path
        break

if csv_drive is None:
    print('ERROR: btc_merged_features.csv not found.')
    print('Check the paths listed above and set csv_drive manually:')
    print("  csv_drive = '/content/drive/MyDrive/YOUR_FOLDER/btc_merged_features.csv'")
else:
    os.makedirs('/content/BTCPredictor2/data', exist_ok=True)
    csv_dest = '/content/BTCPredictor2/data/btc_merged_features.csv'
    if not os.path.exists(csv_dest):
        os.symlink(csv_drive, csv_dest)
    size_gb = os.path.getsize(csv_drive) / 1024**3
    print(f'Found: {csv_drive}')
    print(f'Size:  {size_gb:.1f} GB')
    print('Ready. Run Cell 3 to train TFT.')

In [ ]:
# ============================================================
# CELL 3 - TRAIN TFT  (estimated: 1-2 hours on A100)
# ============================================================

import os, sys, time
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import tensorflow as tf
tf.keras.mixed_precision.set_global_policy('mixed_float16')

import config
config.BATCH_TFT = 512

import pandas as pd
from features.engineer import get_feature_groups
from models import tft_model

groups           = get_feature_groups()
TFT_DYN_FEATURES = groups['tft_dynamic']   # h1/h4/d1 — 30 time-varying features
TFT_STA_FEATURES = groups['tft_static']    # w1/mo1   — 27 static covariates
ALL_TFT          = TFT_DYN_FEATURES + TFT_STA_FEATURES

print(f'TFT dynamic features: {len(TFT_DYN_FEATURES)}  (h1/h4/d1)')
print(f'TFT static features:  {len(TFT_STA_FEATURES)}  (w1/mo1 — macro regime)')
print()

df = pd.read_csv('data/btc_merged_features.csv', parse_dates=['time'])
if df['time'].dt.tz is None:
    df['time'] = pd.to_datetime(df['time'], utc=True)
df = df[df['time'] <= pd.Timestamp(config.TRAINING_CUTOFF_DATE, tz='UTC')].copy()
print(f'Rows after cutoff: {len(df):,}')
print(f'Training TFT  SEQ_LEN={tft_model.SEQ_LEN}  HORIZON={tft_model.HORIZON}min')
print('Progress prints every epoch. Do not close this tab.')
print()

t0 = time.time()
results = tft_model.train(df, TFT_DYN_FEATURES, TFT_STA_FEATURES)
elapsed = time.time() - t0

print()
print('TFT COMPLETE')
print(f'  Test accuracy: {results["test_acc"]:.4f}')
print(f'  Val  accuracy: {results["val_acc"]:.4f}')
print(f'  Time:          {int(elapsed//3600)}h {int((elapsed%3600)//60)}m')
print()
print('Run Cell 4 to train BiLSTM.')

In [ ]:
# ============================================================
# CELL 4 - TRAIN BILSTM  (estimated: 30-45 min on A100)
# ============================================================

import os, sys, time
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import tensorflow as tf
tf.keras.mixed_precision.set_global_policy('mixed_float16')

import config
config.BATCH_BILSTM = 1024

import pandas as pd
from features.engineer import get_feature_groups
from models import bilstm_model

groups          = get_feature_groups()
BILSTM_FEATURES = groups['bilstm']   # 1m/15m/30m — 30 short-term features

print(f'BiLSTM features: {len(BILSTM_FEATURES)}  (1m/15m/30m micro momentum)')
print()

df = pd.read_csv('data/btc_merged_features.csv', parse_dates=['time'])
if df['time'].dt.tz is None:
    df['time'] = pd.to_datetime(df['time'], utc=True)
df = df[df['time'] <= pd.Timestamp(config.TRAINING_CUTOFF_DATE, tz='UTC')].copy()

print(f'Training BiLSTM  SEQ_LEN={bilstm_model.SEQ_LEN}  HORIZON={bilstm_model.HORIZON}min')
print()

t0 = time.time()
results = bilstm_model.train(df, BILSTM_FEATURES)
elapsed = time.time() - t0

print()
print('BiLSTM COMPLETE')
print(f'  Test accuracy: {results["test_acc"]:.4f}')
print(f'  Val  accuracy: {results["val_acc"]:.4f}')
print(f'  Time:          {int(elapsed//3600)}h {int((elapsed%3600)//60)}m')
print()
print('Run Cell 5 to train Meta and download models.')

In [ ]:
# ============================================================
# CELL 5 - TRAIN META + DOWNLOAD  (~5 minutes)
# Models are downloaded as a zip at the end.
# ============================================================

import os, sys, time, shutil, json
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import pandas as pd
import config
from features.engineer import FEATURE_COLS
from models import tft_model, bilstm_model, meta_model

df = pd.read_csv('data/btc_merged_features.csv', parse_dates=['time'])
if df['time'].dt.tz is None:
    df['time'] = pd.to_datetime(df['time'], utc=True)
df = df[df['time'] <= pd.Timestamp(config.TRAINING_CUTOFF_DATE, tz='UTC')].copy()
df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)

n_total   = len(df)
val_start = int(n_total * 0.70)
val_end   = int(n_total * 0.85)
df_val    = df.iloc[val_start:val_end].copy()
val_X     = df_val[FEATURE_COLS].values.astype('float32')

acc_path  = 'models/saved/model_accuracies.json'
saved_acc = json.load(open(acc_path)) if os.path.exists(acc_path) else {}
tft_val_err    = 1.0 - saved_acc.get('tft_val', 0.51)
bilstm_val_err = 1.0 - saved_acc.get('bilstm_val', 0.51)

print('Generating OOF predictions...')
val_tft_probs    = tft_model.predict_proba_batch(val_X, FEATURE_COLS)
val_bilstm_probs = bilstm_model.predict_proba_batch(val_X, FEATURE_COLS)

print('Training XGBoost meta-learner...')
meta_results = meta_model.train(df_val, val_tft_probs, val_bilstm_probs,
                                 tft_val_err, bilstm_val_err, FEATURE_COLS)
w = meta_results.get('weights', {})

print()
print('=' * 50)
print('TRAINING COMPLETE')
print(f'  TFT accuracy:    {saved_acc.get("tft", 0):.3f}  (4h horizon)')
print(f'  BiLSTM accuracy: {saved_acc.get("bilstm", 0):.3f}  (1h horizon)')
print(f'  Meta accuracy:   {meta_results["train_acc"]:.3f}  (1h actionable)')
print(f'  TFT weight:      {w.get("tft", 0):.3f}')
print(f'  BiLSTM weight:   {w.get("bilstm", 0):.3f}')
print('=' * 50)

# Zip the trained models for download
OUT = '/content/models_output'
os.makedirs(OUT, exist_ok=True)
for f in ['tft.keras', 'tft_scaler.pkl', 'bilstm.keras', 'bilstm_scaler.pkl',
          'meta_xgb.pkl', 'model_accuracies.json', 'training_cutoff.txt']:
    src = f'models/saved/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{OUT}/{f}')

shutil.make_archive('/content/btc_models', 'zip', OUT)

# Download the zip to your PC
from google.colab import files
print()
print('Downloading btc_models.zip to your PC...')
files.download('/content/btc_models.zip')
print()
print('Extract the zip and copy all files into your local models/saved/ folder.')
print('Then run: python papertrading/backtest.py --start 2026-04-15')